In [ ]:
import jax
device = 'cuda'  # Change to 'cuda' to use GPU
N_CHAINS = 10
N_STEPS_PER_CHAIN = 10_0000
DoF = 120
hidden_units = 4096

jax.config.update("jax_platform_name", device)

print(jax.devices())

from jax import random
import jax.numpy as jnp
import time
from functools import partial


@partial(jax.jit, static_argnames=("prob_fn",))
def mh_chain_with_all_random_nums(random_values, prob_fn, init_pos):
    # Placeholder implementation of mh_chain

    def mh_kernel(carry, random_values):
        position, old_prob = carry
        proposal = position + (2 * random_values[0] - 1)
        proposal_prob = prob_fn(proposal)
        accept_prob = jnp.minimum(1.0, proposal_prob / old_prob)
        accept = random_values[-1] < accept_prob
        new_position = jnp.where(accept, proposal, position)
        new_prob = jnp.where(accept, proposal_prob, old_prob)
        carry = (new_position, new_prob)
        return carry, new_position

    initial_prob = prob_fn(init_pos)
    carry = (init_pos, initial_prob)
    positions, _ = jax.lax.scan(mh_kernel, carry, random_values)
    return positions


sampler = jax.vmap(mh_chain_with_all_random_nums, in_axes=(0, None, 0))


from flax import linen as nn
from jax import numpy as jnp


# Define a simple MLP using Flax Linen
class MLP(nn.Module):
    architecture: list
    hidden_activation: callable = nn.tanh
    alpha: float = 1.0  # parameter for the final wavefunction

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = nn.Dense(features=self.architecture[i + 1])(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)

        return x

    def get_forward_pass(self):
        string = ""
        for i in range(len(self.architecture) - 1):
            string += f"Dense({self.architecture[i]} -> {self.architecture[i + 1]})"
            if i < len(self.architecture) - 2:
                string += f" -> {self.hidden_activation.__name__} -> "

        return string

model = MLP(architecture=[DoF, hidden_units, 1], hidden_activation=nn.tanh, alpha=1.0)
params = model.init(random.PRNGKey(0), jnp.ones((1, DoF)))

@jax.jit
def prob_fn(x):
    psi = model.apply(params, x).squeeze()
    return jnp.square(psi)


print(model.get_forward_pass())

[CudaDevice(id=0)]
Dense(120 -> 4096) -> tanh -> Dense(4096 -> 1) -> tanh -> 
